In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

import duckdb
import sqlite3

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, recall_score, precision_score, roc_curve, confusion_matrix, auc, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [2]:
conn = duckdb.connect(r'C:\Users\Pichau\OneDrive\Documentos\Estudos Dudu\7. atraso-voos\data\features.duckdb')

In [3]:
df = pd.read_sql_query("SELECT * FROM fs_general_abt", conn)

C:\Users\Pichau\AppData\Local\Temp\ipykernel_13136\1169696912.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query("SELECT * FROM fs_general_abt", conn)


In [4]:
df.head()

,idVoo,flagAtraso,avgPrecipitacaoTotal24h,avgPressaoAtmosferica24h,avgPressaoAtmosfericaMaxUltimaHora24h,avgPressaoAtmosfericaMinUltimaHora24h,avgTemperaturaBulboSeco24h,avgTemperaturaPontoOrvalho24h,avgTemperaturaMaxUltimaHora24h,avgTemperaturaMinUltimaHora24h,...,cancelamentos12h,cancelamentosRota12h,cancelamentosDestino12h,cancelamentosCia12h,cancelamentosCiaRota12h,cancelamentos3h,cancelamentosRota3h,cancelamentosDestino3h,cancelamentosCia3h,cancelamentosCiaRota3h
0,AZU-4082-SBCF-SBKP-202404111950-202404112105,0,0.0,917.504171,917.758334,917.279165,25.125000,17.225000,25.862500,24.529167,...,2,1,1,2,1,1,1,1,1,1
1,TAM-3557-SBCF-SBGR-202404111955-202404112115,0,0.0,917.504171,917.758334,917.279165,25.125000,17.225000,25.862500,24.529167,...,2,0,0,0,0,1,0,0,0,0
2,GLO-1705-SBCF-SBBR-202404111815-202404111935,0,0.0,917.454173,917.708336,917.224998,25.041667,17.058333,25.845833,24.445833,...,2,0,0,0,0,2,0,0,0,0
3,TAM-3555-SBCF-SBGR-202404111455-202404111610,0,0.0,917.291674,917.541669,917.054169,25.020833,16.733333,25.825000,24.441667,...,1,0,0,0,0,0,0,0,0,0
4,AZU-4729-SBCF-SBBE-202404080835-202404081135,1,0.0,920.133334,920.350004,919.929169,22.891667,15.629167,23.679167,21.970833,...,0,0,0,0,0,0,0,0,0,0


# CORRELAÇÃO DE VARIAVEIS

In [5]:
df_target_corr = df.select_dtypes(include='number').corr()['flagAtraso']

In [6]:
df.columns

Index(['idVoo', 'flagAtraso', 'avgPrecipitacaoTotal24h',
       'avgPressaoAtmosferica24h', 'avgPressaoAtmosfericaMaxUltimaHora24h',
       'avgPressaoAtmosfericaMinUltimaHora24h', 'avgTemperaturaBulboSeco24h',
       'avgTemperaturaPontoOrvalho24h', 'avgTemperaturaMaxUltimaHora24h',
       'avgTemperaturaMinUltimaHora24h',
       ...
       'cancelamentos12h', 'cancelamentosRota12h', 'cancelamentosDestino12h',
       'cancelamentosCia12h', 'cancelamentosCiaRota12h', 'cancelamentos3h',
       'cancelamentosRota3h', 'cancelamentosDestino3h', 'cancelamentosCia3h',
       'cancelamentosCiaRota3h'],
      dtype='str', length=144)

In [7]:
df_target_corr.to_frame().sort_values(by='flagAtraso', ascending=False)

,flagAtraso
flagAtraso,1.000000
atrasos12h,0.140750
atrasos24h,0.136322
atrasos3h,0.131169
atrasosCia12h,0.126431
...,...
maxPressaoAtmosferica24h,-0.070557
maxPressaoAtmosfericaMinUltimaHora24h,-0.071209
avgPressaoAtmosfericaMaxUltimaHora24h,-0.072544
avgPressaoAtmosferica24h,-0.072901


# DESBALANCEAMENTO DE CLASSES

In [8]:
df['flagAtraso'].value_counts()[1]/len(df) * 100

np.float64(7.858057598951664)

# ANALISE DE VARIAVEIS CONSTANTES

In [9]:
from sklearn.feature_selection import VarianceThreshold

var_thres = VarianceThreshold(threshold=10)
var_thres.fit(df.drop(columns=['idVoo','flagAtraso']))

var_thres

,"threshold threshold: float, default=0Features with a training-set variance lower than this threshold willbe removed. The default is to keep all features with non-zero variance,i.e. remove the features that have the same value in all samples.",10


In [10]:
var_thres.get_support()

array([False, False, False, False, False,  True, False, False,  True,
        True,  True,  True,  True,  True, False, False, False, False,
       False, False, False,  True,  True, False,  True,  True,  True,
        True,  True,  True, False, False, False,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True, False, False,  True, False, False, False, False,  True,
       False, False,  True,  True,  True,  True,  True,  True, False,
       False,  True, False, False, False,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True, False, False, False,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True, False, False, False, False, False,
       False, False, False, False, False, False, False,  True,  True,
        True,  True, False, False,  True, False, False,  True, False,
        True, False, False,  True, False, False, False, False, False,
       False,  True,

In [11]:
for idx, i in enumerate(var_thres.get_support()):
    if i == False:
        print(df.columns[idx])

idVoo
flagAtraso
avgPrecipitacaoTotal24h
avgPressaoAtmosferica24h
avgPressaoAtmosfericaMaxUltimaHora24h
avgTemperaturaBulboSeco24h
avgTemperaturaPontoOrvalho24h
avgUmidadeRelativa24h
avgVentoDirecaoGraus24h
avgVentoRajadaMax24h
avgVentoVelocidade24h
avgPrecipitacaoTotal12h
avgPressaoAtmosferica12h
avgPressaoAtmosfericaMaxUltimaHora12h
avgTemperaturaPontoOrvalho12h
avgUmidadeRelativa12h
avgVentoDirecaoGraus12h
avgVentoRajadaMax12h
avgUmidadeRelativa3h
avgVentoDirecaoGraus3h
avgVentoVelocidade3h
maxPrecipitacaoTotal24h
maxPressaoAtmosferica24h
maxPressaoAtmosfericaMaxUltimaHora24h
maxTemperaturaBulboSeco24h
maxTemperaturaPontoOrvalho24h
maxUmidadeRelativa24h
maxVentoDirecaoGraus24h
maxVentoVelocidade24h
maxPrecipitacaoTotal12h
maxPressaoAtmosferica12h
maxUmidadeRelativa12h
maxVentoDirecaoGraus12h
maxVentoRajadaMax12h
maxUmidadeRelativa3h
maxVentoDirecaoGraus3h
maxVentoRajadaMax3h
maxVentoVelocidade3h
stddevPrecipitacaoTotal24h
stddevPressaoAtmosferica24h
stddevPressaoAtmosfericaMaxUltima

# MODELING UNDERSAMPLER

In [12]:
X = df.drop(columns=['idVoo','flagAtraso'])
y = df['flagAtraso']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [13]:
len(y_test)

60439

In [14]:
# Se você usou o Undersampling para chegar em 1:3 ou 1:2
model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5,
    eval_metric='AUC', # Foca na métrica que você quer melhorar
    early_stopping_rounds=50,
    verbose=100
)

# Se você NÃO usar undersampling, adicione: auto_class_weights='Balanced'
# Se usar o undersampling, pode deixar sem ou testar um peso leve

model.fit(
    X_train, y_train,               # Dados balanceados (treino)
    eval_set=(X_test, y_test),  # Validação nos dados REAIS (com os 7%)
    use_best_model=True
)

0:	test: 0.5773042	best: 0.5773042 (0)	total: 170ms	remaining: 2m 49s
100:	test: 0.6652995	best: 0.6652995 (100)	total: 3.96s	remaining: 35.2s
200:	test: 0.6721540	best: 0.6721540 (200)	total: 7.38s	remaining: 29.3s
300:	test: 0.6754116	best: 0.6754116 (300)	total: 10.6s	remaining: 24.7s
400:	test: 0.6775553	best: 0.6775553 (400)	total: 13.9s	remaining: 20.8s
500:	test: 0.6792846	best: 0.6793529 (494)	total: 17.1s	remaining: 17s
600:	test: 0.6812000	best: 0.6812000 (600)	total: 20.4s	remaining: 13.5s
700:	test: 0.6828183	best: 0.6828183 (700)	total: 23.8s	remaining: 10.1s
800:	test: 0.6840553	best: 0.6840553 (800)	total: 27.1s	remaining: 6.74s
900:	test: 0.6846456	best: 0.6846659 (868)	total: 30.4s	remaining: 3.34s
999:	test: 0.6848776	best: 0.6850099 (986)	total: 33.7s	remaining: 0us

bestTest = 0.6850099374
bestIteration = 986

Shrink model to first 987 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, l2_leaf_reg=5, learning_rate=0.05, verbose=100)

In [15]:
# Predições
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [16]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, y_proba)
print(f"Novo ROC AUC: {auc}")

Novo ROC AUC: 0.6850099373540038


In [17]:
from imblearn.under_sampling import RandomUnderSampler

# Se você quer que a classe minoritária seja 25% do total (proporção 1:3)
# sampling_strategy = (positivos) / (negativos desejados) -> 14000 / 42000 = 0.33
rus = RandomUnderSampler(sampling_strategy=0.33, random_state=42)

X_res, y_res = rus.fit_resample(X_train, y_train)

print(f"Novo tamanho do treino: {len(X_res)} linhas")

Novo tamanho do treino: 44663 linhas


In [18]:
# Se você usou o Undersampling para chegar em 1:3 ou 1:2
model_under = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=5,
    eval_metric='AUC', # Foca na métrica que você quer melhorar
    early_stopping_rounds=50,
    verbose=100
)

# Se você NÃO usar undersampling, adicione: auto_class_weights='Balanced'
# Se usar o undersampling, pode deixar sem ou testar um peso leve

model_under.fit(
    X_train, y_train,               # Dados balanceados (treino)
    eval_set=(X_res, y_res),  # Validação nos dados REAIS (com os 7%)
    use_best_model=True
)

0:	test: 0.5774963	best: 0.5774963 (0)	total: 46.9ms	remaining: 46.9s
100:	test: 0.6820030	best: 0.6820030 (100)	total: 3.53s	remaining: 31.5s
200:	test: 0.6967637	best: 0.6967637 (200)	total: 6.85s	remaining: 27.2s
300:	test: 0.7085168	best: 0.7085168 (300)	total: 10.1s	remaining: 23.4s
400:	test: 0.7191584	best: 0.7191584 (400)	total: 13.2s	remaining: 19.7s
500:	test: 0.7286714	best: 0.7286714 (500)	total: 16.3s	remaining: 16.2s
600:	test: 0.7392015	best: 0.7392015 (600)	total: 19.5s	remaining: 12.9s
700:	test: 0.7485531	best: 0.7485531 (700)	total: 22.6s	remaining: 9.65s
800:	test: 0.7572488	best: 0.7572488 (800)	total: 25.9s	remaining: 6.43s
900:	test: 0.7651086	best: 0.7651086 (900)	total: 29.1s	remaining: 3.2s
999:	test: 0.7723530	best: 0.7723530 (999)	total: 32.3s	remaining: 0us

bestTest = 0.7723530425
bestIteration = 999



CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, l2_leaf_reg=5, learning_rate=0.05, verbose=100)

In [19]:
# Predições
y_pred_under = model_under.predict(X_test)
y_proba_under = model_under.predict_proba(X_test)[:, 1]

In [20]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_test, y_proba_under)
print(f"Novo ROC AUC: {auc}")

Novo ROC AUC: 0.6848775527342592


In [21]:
# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_under))


Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     55690
           1       0.58      0.03      0.05      4749

    accuracy                           0.92     60439
   macro avg       0.75      0.51      0.51     60439
weighted avg       0.90      0.92      0.89     60439



In [22]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

# 1. Pegue as probabilidades (em vez das classes 0/1)
y_probs = model_under.predict_proba(X_test)[:, 1]

# 2. Teste vários thresholds para encontrar o melhor F1
thresholds = np.linspace(0, 1, 100)
f1_scores = [f1_score(y_test, y_probs >= t) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Melhor Threshold: {best_threshold:.4f}")

# 3. Gere o relatório com o novo threshold
y_pred_novo = (y_probs >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_novo))


Melhor Threshold: 0.1212
              precision    recall  f1-score   support

           0       0.94      0.88      0.91     55690
           1       0.20      0.34      0.25      4749

    accuracy                           0.84     60439
   macro avg       0.57      0.61      0.58     60439
weighted avg       0.88      0.84      0.86     60439



# MDELING PADRAO

In [23]:
modelos = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        class_weight='balanced_subsample'
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        class_weight='balanced'
    ),

    "CatBoost": CatBoostClassifier(
        verbose=0,
        auto_class_weights='Balanced',
        random_state=42
    ),

    "XGBoost": XGBClassifier(
    eval_metric='logloss',
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),
    random_state=42
    ),
}

In [24]:
def avaliar_modelo(modelo, X_train, X_test, y_train, y_test, nome_modelo="Modelo"):
    
    # Treinamento
    modelo.fit(X_train, y_train)
    
    # Predições
    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    
    print(f"\n{'='*60}")
    print(f"AVALIAÇÃO - {nome_modelo}")
    print(f"{'='*60}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    # ROC AUC
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC AUC: {auc:.4f}")
    
    # Matriz de confusão
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
    plt.title(f"Matriz de Confusão [{nome_modelo}]")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.show()
    
    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)

    plt.figure(figsize=(8,6))

    plt.plot(

        fpr,
        tpr,
        label=f"ROC Curve (AUC = {auc:.3f})"

    )

    plt.plot([0,1],[0,1],'k--')

    plt.xlabel("False Positive Rate")

    plt.ylabel("True Positive Rate")

    plt.title(f"Curva ROC [{nome_modelo}]")

    plt.legend()

    plt.grid(alpha=0.3)

    plt.show()

    try:
        importances = pd.Series(modelo.feature_importances_, index=X.columns)
        importances.nlargest(10).plot(kind='barh', title='Top 10 Feature Importance')
        plt.show()
    except:
        print("Modelo não possui feature_importances_")

In [25]:
resultados = []

for nome, modelo in modelos.items():
    resultado = avaliar_modelo(
        modelo,
        X_train,
        X_test,
        y_train,
        y_test,
        nome
    )
    
    resultados.append(resultado)

KeyboardInterrupt: 